# Capstone, a classifier under budget

**Scenario:** a retail assortment desk labels every SKU each night as stock up, hold or mark down.
The pilot ran on the fast model, so finance approved a nightly ceiling. Someone then switched the
default to the reasoning model, to improve quality.

The batch went over the ceiling and came back with fewer labels than before.

Think of it as a pay as you go phone. When the credit runs out the phone still works. It just stops
making calls, and it tells you so.

This capstone joins the two earlier lessons. Here the measurement becomes a control.

## Mechanics

A ceiling is only a control if something checks it before each call. These are the pieces.

| Piece | What it is |
|---|---|
| `ceiling` | Dollars the batch may spend, agreed before it runs |
| `spent` | Dollars already spent, from `cost_of` on every response so far |
| `worst` | The dearest call yet seen on a tier, which is what the next one might cost |
| `tiers` | Models in order of capability, dearest first |
| Rule fallback | A label from ordinary code, costing nothing |
| `finish_reason` of `length` | You paid in full and got nothing back |

The last row is the trap. A truncated reply is not a cheap failure. It is the dearest kind, because
the output count is at its cap.

## The picture

![The budget picks the tier, and the rule catches whatever it cannot afford](images/budget-ceiling.svg)

Every SKU leaves this picture with a label. What changes under pressure is which box produced it.

## The cost

```
spent   = sum of cost_of(usage) for every call so far
headroom = ceiling - spent
afford a tier when headroom >= the dearest call yet seen on it
```

The check runs before the call, not after. Checking after is a report, and a report cannot stop
anything.

## The failure

Eight SKUs, one line each, and a label with three allowed values.

In [1]:
SKUS = [("SKU-1101", "winter parka, north stores, 3 weeks cover, mild forecast"),
        ("SKU-1102", "sun cream, 1 weeks cover, heatwave forecast"),
        ("SKU-1103", "school shoes, 9 weeks cover, term starts in 6 weeks"),
        ("SKU-1104", "garden hose, 14 weeks cover, season ending"),
        ("SKU-1105", "protein bars, 2 weeks cover, steady sales"),
        ("SKU-1106", "fan heater, 11 weeks cover, cold snap forecast"),
        ("SKU-1107", "paddling pool, 20 weeks cover, autumn"),
        ("SKU-1108", "umbrella, 4 weeks cover, wet week forecast")]

SYSTEM = "You are a retail demand planner. Reply with one word: stock_up, hold or mark_down."
LABELS = {"stock_up", "hold", "mark_down"}

One call per SKU. It returns the usage, the label and the reason generation stopped, because that
last one decides whether you got anything for the money.

In [2]:
from vault import Usage, cost_of, get_client, load_env, model_for, summarise

load_env()
FAST, THINKER, SMALL = model_for("default"), model_for("reasoning"), model_for("small")
client = get_client("03-token-economics/03-capstone-a-classifier-under-budget")


def classify(model, note):
    """One SKU, one call. Returns usage, the label text and why it stopped."""
    reply = client.chat.completions.create(
        model=model, max_tokens=300,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": note}])
    choice = reply.choices[0]
    return Usage.from_response(reply), (choice.message.content or "").strip().lower(), \
        choice.finish_reason

The ceiling comes from the pilot, which is the honest way to set one. Two SKUs on the fast model,
scaled to the batch, with headroom the desk agreed to.

In [3]:
HEADROOM = 10                      # what the desk allows above the piloted rate

pilot = [classify(FAST, note)[0] for _, note in SKUS[:2]]
per_sku = summarise(pilot)["usd"] / len(pilot)
CEILING = per_sku * len(SKUS) * HEADROOM

print(f"piloted cost per SKU : ${per_sku:.8f}")
print(f"nightly ceiling      : ${CEILING:.8f} for {len(SKUS)} SKUs")

piloted cost per SKU : $0.00000420
nightly ceiling      : $0.00033600 for 8 SKUs


Now the batch as it shipped, after the default was switched to the reasoning model. No meter, no
check, one call per SKU.

In [4]:
naive, labelled = [], 0

for sku, note in SKUS:
    usage, label, stopped = classify(THINKER, note)
    naive.append(usage)
    labelled += label in LABELS
    print(f"  {sku} out {usage.completion_tokens:4}  ${cost_of(usage):.8f}  {stopped:9} {label!r}")

spend = summarise(naive)["usd"]
print(f"\nspent ${spend:.8f} against a ceiling of ${CEILING:.8f}, labelled {labelled}/{len(SKUS)}")
assert spend <= CEILING, f"the batch spent {spend / CEILING:.1f} times its ceiling"

  SKU-1101 out  256  $0.00010460  length    ''
  SKU-1102 out  197  $0.00008085  stop      'stock_up'
  SKU-1103 out  256  $0.00010460  length    ''
  SKU-1104 out  256  $0.00010440  length    ''
  SKU-1105 out  256  $0.00010440  length    ''
  SKU-1106 out  256  $0.00010445  length    ''
  SKU-1107 out  256  $0.00010445  length    ''
  SKU-1108 out  162  $0.00006685  stop      'stock_up'

spent $0.00077460 against a ceiling of $0.00033600, labelled 2/8


AssertionError: the batch spent 2.3 times its ceiling

## The diagnosis

The assertion fires, and the table above it shows the worse half of the story.

**The ceiling was a number in a document.** Nothing in the loop could read it, so nothing could act
on it. A limit no code checks is a wish.

**The dearest calls returned nothing.** Every row that stopped on `length` burned its whole output
allowance on reasoning and never reached the one word answer. Full price, no label.

**Failure did not stop the batch.** After the first empty reply the loop kept paying the same tier
for the same result, seven more times.

The run overspent and under delivered at once. That pairing is what makes this expensive rather
than annoying.

## The fix

Three pieces. Something that knows what is left, something that costs nothing, and something that
chooses between them before each call.

In [5]:
class Budget:
    """What is left, and whether the next call on a tier can fit inside it."""

    def __init__(self, ceiling):
        self.ceiling, self.spent, self.worst = ceiling, 0.0, {}

    def record(self, model, usage):
        charge = cost_of(usage)
        self.spent += charge
        self.worst[model] = max(self.worst.get(model, 0.0), charge)

    def affords(self, model):
        if self.spent >= self.ceiling:
            return False
        likely = self.worst.get(model, max(self.worst.values(), default=0.0))
        return self.spent + likely <= self.ceiling

An untried tier has no measured worst, so it borrows the dearest call seen anywhere. The first call
of a batch is the one thing not bounded in advance, so the ceiling has to be at least one call wide.

Next the floor. Weeks of cover already decide most of these labels, and that code has no bill.

In [6]:
import re


def rule_label(note):
    """A label from ordinary code. Always available, always free."""
    weeks = int(re.search(r"(\d+) weeks cover", note).group(1))
    if weeks <= 2:
        return "stock_up"
    return "mark_down" if weeks >= 10 else "hold"

Then the choice. Walk the tiers dearest first and take the best one the headroom still covers.

In [7]:
def pick_tier(budget, tiers):
    """The most capable tier the remaining budget covers, or nothing."""
    for model in tiers:
        if budget.affords(model):
            return model
    return None

The batch loop puts them together. A tier that returns nothing usable is dropped, so one bad answer
costs one call rather than the whole run.

In [8]:
def label_batch(skus, tiers, budget):
    """Every SKU gets a label. The budget decides how good that label is."""
    live, out = list(tiers), []
    for sku, note in skus:
        model = pick_tier(budget, live)
        if model is None:
            out.append((sku, rule_label(note), "rule"))
            continue
        usage, label, stopped = classify(model, note)
        budget.record(model, usage)
        if stopped == "stop" and label in LABELS:
            out.append((sku, label, model))
        else:
            live.remove(model)
            out.append((sku, rule_label(note), "rule"))
    return out

Same eight SKUs, same ceiling, same tier order, dearest first.

In [9]:
budget = Budget(CEILING)
result = label_batch(SKUS, [THINKER, FAST, SMALL], budget)

for sku, label, source in result:
    print(f"  {sku} {label:9} from {source}")

print(f"\nbefore: ${spend:.8f} spent, {labelled}/{len(SKUS)} labelled, ceiling broken")
print(f"after : ${budget.spent:.8f} spent, {len(result)}/{len(SKUS)} labelled, "
      f"ceiling ${CEILING:.8f} held")

  SKU-1101 hold      from rule
  SKU-1102 stock_up  from google/gemini-2.5-flash-lite
  SKU-1103 stock_up  from google/gemini-2.5-flash-lite
  SKU-1104 mark_down from google/gemini-2.5-flash-lite
  SKU-1105 hold      from google/gemini-2.5-flash-lite
  SKU-1106 stock_up  from google/gemini-2.5-flash-lite
  SKU-1107 hold      from google/gemini-2.5-flash-lite
  SKU-1108 stock_up  from google/gemini-2.5-flash-lite

before: $0.00077460 spent, 2/8 labelled, ceiling broken
after : $0.00013470 spent, 8/8 labelled, ceiling $0.00033600 held


## The gate

The property worth protecting is the hardest one to notice going wrong. With no money at all, every
SKU still gets a label and nothing is charged. This test calls no model.

In [10]:
def test_a_dead_budget_still_labels_every_sku():
    broke = Budget(0.0)
    out = label_batch(SKUS, [THINKER, FAST, SMALL], broke)
    assert len(out) == len(SKUS), "a SKU left the batch with no label"
    assert broke.spent == 0.0, f"spent {broke.spent} against a ceiling of zero"


test_a_dead_budget_still_labels_every_sku()
print("gate holds: zero budget, every SKU labelled, nothing spent")

gate holds: zero budget, every SKU labelled, nothing spent


Return `True` from `affords` when the ceiling is already reached and this test spends real money.

### Enterprise exploration

- The ceiling here is one batch in one process. Where does `spent` live when four workers share a
  nightly budget, and what stops two of them spending the last dollar?
- The rule fallback is cheaper and worse. Who is told that tonight's labels came from ordinary code,
  and what does a week of nobody noticing cost?
- The first call on an untried tier is not bounded in advance. What would you cap it with, and what
  do you give up by capping it?
- A buyer disputes a mark down. What has to be written down at the moment of the call for you to
  say, a month later, which tier produced it and what it cost?

### Key takeaways

- A ceiling nothing reads before a call is a wish, not a control.
- Measure spend from real responses. An estimate cannot enforce anything.
- A truncated reply is the dearest failure there is. Drop the tier that produced it.
- Degrading means every item still gets an answer, from a worse and cheaper source.